<a href="https://colab.research.google.com/github/aditya456723/AI-virtual-assistant-python/blob/master/Ai_avator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Clone SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker

Cloning into 'SadTalker'...
remote: Enumerating objects: 1605, done.
remote: Counting objects: 100% (551/551), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 1605 (delta 474), reused 449 (delta 449), pack-reused 1054 (from 1)
Receiving objects: 100% (1605/1605), 92.20 MiB | 16.21 MiB/s, done.
Resolving deltas: 100% (882/882), done.
/content/SadTalker


In [4]:
# Cell 2 - Install dependencies (fixed for modern Colab 2025/2026)
# Skip torch install - Colab already has a working version
import torch
print("Torch version already installed:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

# Only install missing packages
!pip install -q numpy
!pip install -q face-alignment
!pip install -q imageio
!pip install -q imageio-ffmpeg
!pip install -q librosa
!pip install -q numba
!pip install -q pydub
!pip install -q scipy
!pip install -q kornia
!pip install -q yacs
!pip install -q einops
!pip install -q safetensors
!pip install -q basicsr
!pip install -q facexlib
!pip install -q realesrgan

Torch version already installed: 2.10.0+cu128
CUDA: True


In [7]:
# Cell 3 - Use official download script from SadTalker repo
import os
os.chdir("/content/SadTalker")

# Delete empty files
!rm -f checkpoints/* gfpgan/weights/*

# Use the official bash script included in SadTalker repo
!bash scripts/download_models.sh

mkdir: cannot create directory ‘./checkpoints’: File exists
--2026-03-20 09:29:29--  https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/569518584/ccc415aa-c6f4-47ee-8250-b10bf440ba62?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-20T10%3A27%3A41Z&rscd=attachment%3B+filename%3Dmapping_00109-model.pth.tar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-20T09%3A27%3A09Z&ske=2026-03-20T10%3A27%3A41Z&sks=b&skv=2018-11-09&sig=BmWbM1qIvJ%2FV7qKj2dwzG5ol7jSViXVKQwSYKXBblHk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5

In [8]:
# Cell 4 - Verify GPU
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

Fri Mar 20 09:30:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# Cell 5 - Upload photo + audio
from google.colab import files
import shutil, os

os.makedirs("/content/inputs", exist_ok=True)

print("Upload your PHOTO:")
up = files.upload()
for f in up:
    shutil.move(f, f"/content/inputs/{f}")
    photo = f"/content/inputs/{f}"

print("Upload your AUDIO (.wav):")
up = files.upload()
for f in up:
    shutil.move(f, f"/content/inputs/{f}")
    audio = f"/content/inputs/{f}"

print("Photo:", photo)
print("Audio:", audio)

Upload your PHOTO:


Saving WhatsApp Image 2026-03-20 at 13.10.44.jpeg to WhatsApp Image 2026-03-20 at 13.10.44.jpeg
Upload your AUDIO (.wav):


Saving audio.wav to audio.wav
Photo: /content/inputs/WhatsApp Image 2026-03-20 at 13.10.44.jpeg
Audio: /content/inputs/audio.wav


In [11]:
# Fix the numpy compatibility issue
!sed -i 's/np.VisibleDeprecationWarning/DeprecationWarning/g' \
  /content/SadTalker/src/face3d/util/preprocess.py

# Verify fix
!grep -n "DeprecationWarning" /content/SadTalker/src/face3d/util/preprocess.py

12:warnings.filterwarnings("ignore", category=DeprecationWarning) 


In [13]:
# Fix all numpy 2.x issues in SadTalker codebase
!grep -r "np.VisibleDeprecationWarning" /content/SadTalker --include="*.py" -l

!find /content/SadTalker -name "*.py" -exec \
  sed -i 's/np\.VisibleDeprecationWarning/DeprecationWarning/g' {} \;

!find /content/SadTalker -name "*.py" -exec \
  sed -i 's/np\.bool\b/bool/g' {} \;

!find /content/SadTalker -name "*.py" -exec \
  sed -i 's/np\.int\b/int/g' {} \;

!find /content/SadTalker -name "*.py" -exec \
  sed -i 's/np\.float\b/float/g' {} \;

!find /content/SadTalker -name "*.py" -exec \
  sed -i 's/np\.complex\b/complex/g' {} \;

print("All numpy 2.x fixes applied!")

All numpy 2.x fixes applied!


In [15]:
# Fix basicsr torchvision compatibility
!pip install basicsr==1.4.2 -q
!pip install facexlib==0.3.0 -q
!pip install gfpgan==1.3.8 -q

# Fix the broken import directly
!pip show torchvision | grep Version

Version: 0.25.0+cu128


In [17]:
# Direct patch - replace the broken import
import site
site_packages = site.getsitepackages()[0]

file_path = f"{site_packages}/basicsr/data/degradations.py"

with open(file_path, "r") as f:
    content = f.read()

# Fix the broken import
content = content.replace(
    "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
    "from torchvision.transforms.functional import rgb_to_grayscale"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patched!")

Patched!


In [19]:
# Patch the problematic line
file_path = "/content/SadTalker/src/face3d/util/preprocess.py"

with open(file_path, "r") as f:
    content = f.read()

# Fix inhomogeneous array issue
content = content.replace(
    "trans_params = np.array([w0, h0, s, t[0], t[1]])",
    "trans_params = np.array([w0, h0, s, float(t[0]), float(t[1])])"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patched!")

# Verify
!grep -n "trans_params" /content/SadTalker/src/face3d/util/preprocess.py

Patched!
101:    trans_params = np.array([w0, h0, s, float(t[0]), float(t[1])])
103:    return trans_params, img_new, lm_new, mask_new


In [20]:
import os
os.chdir("/content/SadTalker")
os.makedirs("/content/results", exist_ok=True)

!python inference.py \
  --driven_audio "{audio}" \
  --source_image "{photo}" \
  --result_dir "/content/results" \
  --still \
  --preprocess full \
  --enhancer gfpgan \
  --size 256


using safetensor as default
3DMM Extraction for source image
landmark Det:: 100% 1/1 [00:00<00:00,  6.98it/s]
3DMM Extraction In Video:: 100% 1/1 [00:00<00:00,  9.28it/s]
mel:: 100% 347/347 [00:00<00:00, 51907.11it/s]
audio2exp:: 100% 35/35 [00:00<00:00, 295.62it/s]
Face Renderer:: 100% 174/174 [01:47<00:00,  1.61it/s]
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (256, 251) to (256, 256) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
The generated video is named /content/results/2026_03_20_09.36.53/WhatsApp Image 2026-03-20 at 13.10.44##audio.mp4
OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
seamlessClone:: 100% 347/347 [03:48<00:00,  1.52it/s]
The generated video is n

In [21]:
from google.colab import files
import glob

# Find all generated videos
results = glob.glob("/content/results/**/*.mp4", recursive=True)
for r in results:
    print(r)

/content/results/2026_03_20_09.36.53/temp_WhatsApp Image 2026-03-20 at 13.10.44##audio.mp4
/content/results/2026_03_20_09.36.53/WhatsApp Image 2026-03-20 at 13.10.44##audio.mp4
/content/results/2026_03_20_09.36.53/temp_WhatsApp Image 2026-03-20 at 13.10.44##audio_enhanced.mp4
/content/results/2026_03_20_09.36.53/WhatsApp Image 2026-03-20 at 13.10.44##audio_full.mp4


In [22]:
from google.colab import files

# Best quality - full image with enhancement
files.download("/content/results/2026_03_20_09.36.53/WhatsApp Image 2026-03-20 at 13.10.44##audio_full.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>